# Few-shot transfer: Madrid to Amsterdam

This clean submission notebook uses the fixed-seed implementation in `5-Improved_Transfer.py`. It trains the Madrid Stage 1 prior, adapts to a configurable Amsterdam label budget, and evaluates only the disjoint Amsterdam query pixels.

The four presentation budgets are 5, 25, 100, and 200 labels per class. The code accepts any comma-separated budget through `SAMPLE_SIZES`.


In [ ]:
from pathlib import Path
import os, subprocess, sys

PROJECT_DIR = Path.cwd()
if not (PROJECT_DIR / '5-Improved_Transfer.py').exists():
    PROJECT_DIR = Path.home() / 'Building_Age_Classification_Submission'
os.chdir(PROJECT_DIR)
SAMPLE_SIZES = [5, 25, 100, 200]
N_TRIALS = 10
RANDOM_SEED = 42
OUTPUT_DIR = PROJECT_DIR / 'results'
print('Project:', PROJECT_DIR)
print('Budgets:', SAMPLE_SIZES, 'labels per class; seed:', RANDOM_SEED)


## Reproduce training and evaluation

Run this cell to execute the end-to-end pipeline. It uses a fixed seed and saves the fitted Stage 1 artifact plus a results CSV and F1-versus-log2(n) plot. Install the packages in `requirements.txt` first. The current scored run is summarized below; re-running reproduces the evaluation and regenerates the artifacts.


In [ ]:
cmd = [
    sys.executable, '5-Improved_Transfer.py',
    '--shots', ','.join(map(str, SAMPLE_SIZES)),
    '--trials', str(N_TRIALS), '--seed', str(RANDOM_SEED),
    '--out-dir', str(OUTPUT_DIR),
]
subprocess.run(cmd, cwd=PROJECT_DIR, check=True)


## Recorded result summary

These values are from the already completed 10-trial random-pixel run. The Madrid figure is the source-domain cross-validation reference.

| Evaluation | Labels per class | Macro F1 mean | Standard deviation |
|---|---:|---:|---:|
| Madrid source CV | — | 0.6281 | 0.0043 |
| Amsterdam random-pixel | 5 | 0.6335 | 0.0100 |
| Amsterdam random-pixel | 25 | 0.6646 | 0.0090 |
| Amsterdam random-pixel | 100 | 0.7153 | 0.0070 |
| Amsterdam random-pixel | 200 | 0.7316 | 0.0047 |

The separate spatial-block evaluation is not mixed into this table: it scored 0.7020 ± 0.0280 at 200 labels/class. The random-pixel setup can be optimistic because nearby pixels may occur in both support and query sets; neighbourhood features use full-scene unlabeled imagery.


In [ ]:
import pandas as pd
from IPython.display import display, Image
display(pd.read_csv(OUTPUT_DIR / 'amsterdam_f1_results.csv'))
display(Image(filename=str(OUTPUT_DIR / 'amsterdam_f1_vs_log2_n.png')))
print('Stage 1 model:', OUTPUT_DIR / 'stage1_madrid_model.joblib')
